## Transform Drivers Data
- Read bronze drivers table
- Keep only the columns required for analytics (drop url column)
- Standardise column names using snal case(driverId -> driver_id,dateOfbirth -> date_of_birth)
- Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case
- Remove duplicate records
- Transform values of columns nationality to Title Case
- Write the transformed data to silver drivers table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.drivers'
silver_table = f'{catalog_name}.{silver_schema}.drivers'

###Step 1 - Read bronze circuits table

In [0]:
drivers_df = spark.table(bronze_table)
display(drivers_df)

### Step 2 - Keep only the columns required for analytics (drop url column)

In [0]:

from pyspark.sql import functions as F


In [0]:

drivers_dropped_df = drivers_df.drop('url')
display(drivers_dropped_df)

### Step 3
- Standardise column names using snal case(circuitId -> circuit_id)
- Rename columns to make them more meaningful(lat -> lattitude)

In [0]:
drivers_renamed_df = ( drivers_dropped_df
    .withColumnsRenamed(
        {
        'driverId':'driver_id',
        'dateOfBirth':'date_of_birth'
        }
        )
)


In [0]:
drivers_concatenated_df = drivers_renamed_df\
.withColumn('driver_name',F.initcap(F.concat_ws(' ', F.col('name.givenName'),F.col('name.familyName')) ) )\
    .drop(F.col('name'))
display(drivers_concatenated_df)

### Step 6 - Remove duplicate records

In [0]:

#USING dropDuplicates(we can pass columns if needed) METHOD
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

display(drivers_distinct_df)


### Step 7 -Transform values of columns circuit_name and locality to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
    )
display(drivers_final_df)

### Step 8 - Write the transformed data to silver circuits table


In [0]:
(
    drivers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)

)

In [0]:
display(spark.table(silver_table))